<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

[ваш текст]

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) исользуйтие в проекте коллекции, делегаты, события.


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [2]:
using System;
using System.Collections.Generic;
using System.Linq;

public interface ITrackable
{
    string TrackNumber { get; set; }
    void Track();
}

public interface IInsurable
{
    decimal InsuranceCost { get; set; }
    void Insure();
}

// ================= ДЕЛЕГАТЫ =================
public delegate void DeliveryLogHandler(string message);
public delegate bool DeliveryFilter(ShippingOption option);

// Базовый класс
public class ShippingOption
{
    private int _id;
    private string _name;
    private decimal _cost;
    private string _carrier;
    private double _weightKg;
    private bool _isFragile;

    // Новые атрибуты
    private string _originCity;
    private DateTime _createdAt;
    private int _priorityLevel;

    public int Id { get => _id; set => _id = value; }
    public string Name { get => _name; set => _name = value; }
    public decimal Cost { get => _cost; set => _cost = value; }
    public string Carrier { get => _carrier; set => _carrier = value; }
    public double WeightKg { get => _weightKg; set => _weightKg = value; }
    public bool IsFragile { get => _isFragile; set => _isFragile = value; }

    public string OriginCity { get => _originCity; set => _originCity = value; }
    public DateTime CreatedAt { get => _createdAt; set => _createdAt = value; }
    public int PriorityLevel { get => _priorityLevel; set => _priorityLevel = value; }

    // Событие
    public event DeliveryLogHandler StatusChanged;

    // Защищённый метод для вызова события (паттерн .NET)
    protected void OnStatusChanged(string message)
    {
        StatusChanged?.Invoke(message);
    }

    public ShippingOption(int id, string name, decimal cost, string carrier, double weightKg, bool isFragile = false,
                          string originCity = "Москва", int priorityLevel = 1)
    {
        _id = id;
        _name = name;
        _cost = cost;
        _carrier = carrier;
        _weightKg = weightKg;
        _isFragile = isFragile;
        _originCity = originCity;
        _priorityLevel = priorityLevel;
        _createdAt = DateTime.Now;
    }

    public virtual decimal CalculateCost() => Cost;
    public virtual string EstimateDeliveryTime() => "Время не указано";
    public virtual string GetDetails() => $"ID: {Id}, {Name}, {Cost} руб., Перевозчик: {Carrier}, Вес: {WeightKg} кг, Хрупкий: {IsFragile}, Откуда: {OriginCity}, Приоритет: {PriorityLevel}";
    public virtual void PrintLabel() => Console.WriteLine($"Этикетка: [{Id}] {Name} ({Carrier})");

    public void PrintLabel(bool detailed)
    {
        if (detailed)
            Console.WriteLine($"Этикетка (подробно): [{Id}] {Name} | {Carrier} | {WeightKg} кг | {CalculateCost()} руб.");
        else
            PrintLabel();
    }

    public string GetDetails(string prefix) => $"{prefix}: {GetDetails()}";

    public virtual decimal CalculateDiscount() => 0;

    // Новые методы
    public void ChangePriority(int newPriority)
    {
        PriorityLevel = newPriority;
        OnStatusChanged($"[{Name}] Приоритет изменён на {newPriority}");
    }

    public bool IsRecentlyCreated(int hours = 24) => (DateTime.Now - CreatedAt).TotalHours <= hours;

    public string GetShortSummary() => $"{Id}: {Name} — {CalculateCost()} руб. ({Carrier}, приоритет {PriorityLevel})";
}

// Производный класс StandardDelivery
public class StandardDelivery : ShippingOption
{
    public int AvgDays { get; set; }
    public bool IsTrackable { get; set; }
    public string DeliveryZone { get; set; }

    // Новые атрибуты
    public string PackagingType { get; set; }
    public bool IsEcoFriendly { get; set; }
    public int MaxAttempts { get; set; }

    public StandardDelivery(int id, string name, decimal cost, string carrier, double weightKg,
                            int days, bool isTrackable, string deliveryZone = "Город", bool isFragile = false,
                            string packagingType = "Картон", bool isEcoFriendly = true, int maxAttempts = 3,
                            string originCity = "Москва", int priorityLevel = 1)
        : base(id, name, cost, carrier, weightKg, isFragile, originCity, priorityLevel)
    {
        AvgDays = days;
        IsTrackable = isTrackable;
        DeliveryZone = deliveryZone;
        PackagingType = packagingType;
        IsEcoFriendly = isEcoFriendly;
        MaxAttempts = maxAttempts;
    }

    public override string EstimateDeliveryTime() => $"Среднее время: {AvgDays} дней";
    public override string GetDetails() => base.GetDetails() + $", {AvgDays} дней, Зона: {DeliveryZone}, Упаковка: {PackagingType}, Эко: {IsEcoFriendly}, Попыток: {MaxAttempts}";

    public void NotifyClient() => Console.WriteLine($"[{Name}] Клиент уведомлён о стандартной доставке");
    public void NotifyClient(string message) => Console.WriteLine($"[{Name}] Уведомление клиенту: {message}");

    public override decimal CalculateDiscount() => IsFragile ? Cost * 0.05m : 0;

    // Новые методы
    public void IncreaseAttempts() => MaxAttempts++;

    public bool IsEcoPackaging() => IsEcoFriendly && PackagingType == "Биоразлагаемая";

    public void NotifyClient(int delayMinutes) =>
        Console.WriteLine($"[{Name}] Клиент будет уведомлён через {delayMinutes} мин. (попыток: {MaxAttempts})");
}

// Производный класс ExpressDelivery
public class ExpressDelivery : ShippingOption
{
    public int MinHours { get; set; }
    public decimal Surcharge { get; set; }
    public bool HasPriority { get; set; }
    public string VehicleType { get; set; }

    // Новые атрибуты
    public bool RequiresSignature { get; set; }
    public decimal FuelSurcharge { get; set; }
    public string DriverName { get; set; }

    public ExpressDelivery(int id, string name, decimal cost, string carrier, double weightKg,
                           int hours, decimal surcharge, bool hasPriority, string vehicleType = "Фургон", bool isFragile = false,
                           bool requiresSignature = true, decimal fuelSurcharge = 50, string driverName = "Иванов",
                           string originCity = "Москва", int priorityLevel = 2)
        : base(id, name, cost, carrier, weightKg, isFragile, originCity, priorityLevel)
    {
        MinHours = hours;
        Surcharge = surcharge;
        HasPriority = hasPriority;
        VehicleType = vehicleType;
        RequiresSignature = requiresSignature;
        FuelSurcharge = fuelSurcharge;
        DriverName = driverName;
    }

    public override decimal CalculateCost() => Cost + Surcharge + FuelSurcharge;
    public override string EstimateDeliveryTime() => $"Минимальное время: {MinHours} часов";
    public override string GetDetails() =>
        base.GetDetails() + $", наценка {Surcharge} руб., Транспорт: {VehicleType}, Подпись: {RequiresSignature}, Топливо: {FuelSurcharge}, Водитель: {DriverName}";

    public void SendSms() => Console.WriteLine($"[{Name}] SMS отправлено клиенту");
    public void SendSms(string phone) => Console.WriteLine($"[{Name}] SMS на номер {phone} отправлено");

    // Новые методы
    public void AssignDriver(string driver) => DriverName = driver;

    public bool NeedsSignature() => RequiresSignature;

    public void SendSms(string phone, int delayMinutes) =>
        Console.WriteLine($"[{Name}] SMS на {phone} будет отправлено через {delayMinutes} мин. (водитель: {DriverName})");
}

// Производный класс Pickup
public class Pickup : ShippingOption
{
    public string Address { get; set; }
    public string WorkingHours { get; set; }
    public int Capacity { get; set; }

    // Новые атрибуты
    public bool HasParking { get; set; }
    public string ContactPerson { get; set; }
    public int QueueLength { get; set; }

    public Pickup(int id, string name, decimal cost, string carrier, double weightKg,
                  string address, string workingHours, int capacity = 50,
                  bool hasParking = true, string contactPerson = "Петров", int queueLength = 0,
                  string originCity = "Москва", int priorityLevel = 1)
        : base(id, name, cost, carrier, weightKg, false, originCity, priorityLevel)
    {
        Address = address;
        WorkingHours = workingHours;
        Capacity = capacity;
        HasParking = hasParking;
        ContactPerson = contactPerson;
        QueueLength = queueLength;
    }

    public override decimal CalculateCost() => 0;
    public override string EstimateDeliveryTime() => "Самовывоз — доступно сразу";
    public override string GetDetails() =>
        $"ID: {Id}, {Name}, Адрес: {Address}, Вместимость: {Capacity} чел., Парковка: {HasParking}, Контакт: {ContactPerson}, Очередь: {QueueLength}";

    public void GiveDirections() => Console.WriteLine($"[{Name}] Схема проезда к {Address}");
    public void GiveDirections(bool withMap) => Console.WriteLine($"[{Name}] Схема проезда к {Address}{(withMap ? " (с картой)" : "")}");

    // Новые методы
    public void AddToQueue() => QueueLength++;

    public bool HasFreeCapacity() => QueueLength < Capacity;

    public void GiveDirections(string contactPerson) =>
        Console.WriteLine($"[{Name}] Схема проезда к {Address}, контакт: {contactPerson}");
}

// Производный класс RobotCourierDelivery
public class RobotCourierDelivery : ExpressDelivery
{
    public double MaxRangeKm { get; set; }
    public bool IsAutonomous { get; set; }
    public string BatteryType { get; set; }

    // Новые атрибуты
    public int BatteryLevel { get; set; }
    public bool IsCharging { get; set; }
    public string FirmwareVersion { get; set; }

    public RobotCourierDelivery(int id, string name, decimal cost, string carrier, double weightKg,
                                int hours, decimal surcharge, bool hasPriority,
                                double maxRangeKm, bool isAutonomous, string batteryType = "Li-Ion",
                                int batteryLevel = 100, bool isCharging = false, string firmwareVersion = "1.0.0",
                                string originCity = "Москва", int priorityLevel = 3)
        : base(id, name, cost, carrier, weightKg, hours, surcharge, hasPriority, "Робот", false,
               true, 30, "Робот-1", originCity, priorityLevel)
    {
        MaxRangeKm = maxRangeKm;
        IsAutonomous = isAutonomous;
        BatteryType = batteryType;
        BatteryLevel = batteryLevel;
        IsCharging = isCharging;
        FirmwareVersion = firmwareVersion;
    }

    public override string GetDetails() =>
        base.GetDetails() + $", дальность: {MaxRangeKm} км, батарея: {BatteryType}, заряд: {BatteryLevel}%, прошивка: {FirmwareVersion}";

    public void ActivateRobot() => Console.WriteLine($"[{Name}] Робот-курьер активирован");
    public void ActivateRobot(double range) => Console.WriteLine($"[{Name}] Робот активирован с лимитом {range} км");

    // Новые методы
    public void DrainBattery(int percent) => BatteryLevel = Math.Max(0, BatteryLevel - percent);

    public void StartCharging() => IsCharging = true;

    public void ActivateRobot(double range, int minBattery) =>
        Console.WriteLine($"[{Name}] Робот активирован с лимитом {range} км (мин. заряд: {minBattery}%, текущий: {BatteryLevel}%)");
}

// КЛАСС С ЯВНОЙ РЕАЛИЗАЦИЕЙ ИНТЕРФЕЙСОВ
public class TrackedExpress : ExpressDelivery, ITrackable, IInsurable
{
    private string _trackNumber;
    private decimal _insuranceCost;
    public string Currency { get; set; }

    // Новые атрибуты
    public string LastLocation { get; set; }
    public DateTime LastUpdateTime { get; set; }
    public bool IsDelivered { get; set; }

    string ITrackable.TrackNumber { get => _trackNumber; set => _trackNumber = value; }
    decimal IInsurable.InsuranceCost { get => _insuranceCost; set => _insuranceCost = value; }

    public TrackedExpress(int id, string name, decimal cost, string carrier, double weightKg,
                          int hours, decimal surcharge, bool hasPriority,
                          string trackNumber, decimal insuranceCost, string currency = "RUB",
                          string lastLocation = "Склад", bool isDelivered = false,
                          string originCity = "Москва", int priorityLevel = 2)
        : base(id, name, cost, carrier, weightKg, hours, surcharge, hasPriority, "Фургон", false,
               true, 50, "Сидоров", originCity, priorityLevel)
    {
        _trackNumber = trackNumber;
        _insuranceCost = insuranceCost;
        Currency = currency;
        LastLocation = lastLocation;
        LastUpdateTime = DateTime.Now;
        IsDelivered = isDelivered;
    }

    void ITrackable.Track() => Console.WriteLine($"[{Name}] Отслеживание по номеру: {_trackNumber} (валюта: {Currency})");
    void IInsurable.Insure() => Console.WriteLine($"[{Name}] Застраховано на {_insuranceCost} {Currency}");

    public override decimal CalculateCost() => base.CalculateCost() + _insuranceCost;
    public override string GetDetails() => base.GetDetails() + $", трек: {_trackNumber}, страховка: {_insuranceCost} {Currency}, локация: {LastLocation}, доставлено: {IsDelivered}";

    // Новые методы
    public void UpdateLocation(string location)
    {
        LastLocation = location;
        LastUpdateTime = DateTime.Now;
        OnStatusChanged($"[{Name}] Местоположение обновлено: {location}");
    }

    public void MarkDelivered()
    {
        IsDelivered = true;
        OnStatusChanged($"[{Name}] Доставка завершена");
    }

    public void UpdateLocation(string location, string comment) =>
        Console.WriteLine($"[{Name}] Локация: {location} ({comment}), время: {DateTime.Now:HH:mm}");
}

// КЛАСС ДЛЯ УПРАВЛЕНИЯ ЗАВИСИМОСТЯМИ
public class DeliveryService
{
    private readonly ShippingOption _delivery;

    public DeliveryService(ShippingOption delivery)
    {
        _delivery = delivery;
    }

    public void ProcessDelivery()
    {
        Console.WriteLine($"--- Обработка службой доставки ---");
        Console.WriteLine(_delivery.GetDetails());
        Console.WriteLine($"Итоговая стоимость: {_delivery.CalculateCost()} руб.");
        Console.WriteLine($"Время: {_delivery.EstimateDeliveryTime()}");

        if (_delivery is ITrackable trackable)
        {
            trackable.Track();
        }
        if (_delivery is IInsurable insurable)
        {
            insurable.Insure();
        }
        Console.WriteLine();
    }
}

// ================= ДЕМОНСТРАЦИЯ =================
var deliveries = new ShippingOption[]
{
    new StandardDelivery(1, "Стандартная", 300, "Почта России", 2.5, 5, true, "Область", true),
    new ExpressDelivery(2, "Экспресс", 500, "СДЭК", 1.0, 6, 300, true, "Мотоцикл"),
    new Pickup(3, "Самовывоз", 0, "Своими силами", 0, "Москва, ул. Ленина, 10", "09:00–21:00", 30),
    new RobotCourierDelivery(4, "Робот-курьер", 800, "RoboExpress", 0.5, 2, 400, true, 15.0, true, "Graphene"),
    new TrackedExpress(5, "Отслеживаемый экспресс", 600, "DHL", 1.5, 4, 250, true, "TRK-987654", 150, "USD")
};

Console.WriteLine("=== Тестирование управления зависимостями (Dependency Injection) ===");
foreach (var d in deliveries)
{
    DeliveryService service = new DeliveryService(d);
    service.ProcessDelivery();
}

Console.WriteLine("=== Проверка явной реализации интерфейсов ===");
TrackedExpress tracked = (TrackedExpress)deliveries[4];
ITrackable trackable = tracked;
IInsurable insurable = tracked;

trackable.Track();
insurable.Insure();

// ================= КОЛЛЕКЦИИ =================

Console.WriteLine();
Console.WriteLine("=== Коллекции ===");

// List<T> — все доставки
List<ShippingOption> deliveryList = new List<ShippingOption>(deliveries);
Console.WriteLine($"List<ShippingOption>: {deliveryList.Count} элементов");

// Dictionary<TKey, TValue> — по ID
Dictionary<int, ShippingOption> deliveryDict = new Dictionary<int, ShippingOption>();
foreach (var d in deliveryList)
{
    deliveryDict[d.Id] = d;
}
Console.WriteLine($"Dictionary<int, ShippingOption>: {deliveryDict.Count} элементов");
Console.WriteLine($"Поиск по ID=3: {deliveryDict[3].GetShortSummary()}");

// HashSet<T> — уникальные перевозчики
HashSet<string> carriers = new HashSet<string>();
foreach (var d in deliveryList)
{
    carriers.Add(d.Carrier);
}
Console.WriteLine($"HashSet<string> перевозчиков: {string.Join(", ", carriers)}");

// ================= ДЕЛЕГАТЫ =================

Console.WriteLine();
Console.WriteLine("=== Делегаты ===");

// Делегат для логирования
DeliveryLogHandler log = message => Console.WriteLine($"[LOG] {message}");

// Делегат для фильтрации — только доставки дороже 400 руб.
DeliveryFilter expensiveFilter = option => option.CalculateCost() > 400;

// Применяем делегаты
foreach (var d in deliveryList)
{
    if (expensiveFilter(d))
    {
        log($"Дорогая доставка: {d.GetShortSummary()}");
    }
}

// ================= СОБЫТИЯ =================

Console.WriteLine();
Console.WriteLine("=== События ===");

// Подписываемся на события
foreach (var d in deliveryList)
{
    d.StatusChanged += message => Console.WriteLine($"[СОБЫТИЕ] {message}");
}

// Вызываем события через новые методы
deliveries[0].ChangePriority(5);
deliveries[3].ChangePriority(1);
tracked.UpdateLocation("Сортировочный центр");
tracked.MarkDelivered();

=== Тестирование управления зависимостями (Dependency Injection) ===
--- Обработка службой доставки ---
ID: 1, Стандартная, 300 руб., Перевозчик: Почта России, Вес: 2,5 кг, Хрупкий: True, Откуда: Москва, Приоритет: 1, 5 дней, Зона: Область, Упаковка: Картон, Эко: True, Попыток: 3
Итоговая стоимость: 300 руб.
Время: Среднее время: 5 дней

--- Обработка службой доставки ---
ID: 2, Экспресс, 500 руб., Перевозчик: СДЭК, Вес: 1 кг, Хрупкий: False, Откуда: Москва, Приоритет: 2, наценка 300 руб., Транспорт: Мотоцикл, Подпись: True, Топливо: 50, Водитель: Иванов
Итоговая стоимость: 850 руб.
Время: Минимальное время: 6 часов

--- Обработка службой доставки ---
ID: 3, Самовывоз, Адрес: Москва, ул. Ленина, 10, Вместимость: 30 чел., Парковка: True, Контакт: Петров, Очередь: 0
Итоговая стоимость: 0 руб.
Время: Самовывоз — доступно сразу

--- Обработка службой доставки ---
ID: 4, Робот-курьер, 800 руб., Перевозчик: RoboExpress, Вес: 0,5 кг, Хрупкий: False, Откуда: Москва, Приоритет: 3, наценка 400 р